In [32]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Annotated, List

import random

import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.grammar import extract_grammar
from geneticengine.grammar.decorators import weight
from geneticengine.problems import SingleObjectiveProblem, MultiObjectiveProblem
from geneticengine.random.sources import NativeRandomSource
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.evaluation.budget import TimeBudget, EvaluationBudget
from geneticengine.representations.tree.initializations import MaxDepthDecider, FullDecider, ProgressivelyTerminalDecider, PositionIndependentGrowDecider
from geneticengine.representations.tree.operators import GrowInitializer, PositionIndependentGrowInitializer, FullInitializer, RampedHalfAndHalfInitializer
from geneticengine.algorithms.gp.operators.initializers import HalfAndHalfInitializer, StandardInitializer
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.representations.grammatical_evolution.structured_ge import StructuredGrammaticalEvolutionRepresentation
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker
from geneticengine.evaluation.parallel import ParallelEvaluator

from geneticengine.algorithms.gp.operators.combinators import ParallelStep, SequenceStep
from geneticengine.algorithms.gp.operators.crossover import GenericCrossoverStep
from geneticengine.algorithms.gp.operators.elitism import ElitismStep
from geneticengine.algorithms.gp.operators.mutation import GenericMutationStep
from geneticengine.algorithms.gp.operators.novelty import NoveltyStep
from geneticengine.algorithms.gp.operators.selection import LexicaseSelection, TournamentSelection

from geneticengine.solutions.individual import Individual, PhenotypicIndividual
from geneticengine.algorithms.gp.structure import GeneticStep
from geneticengine.problems import Problem
from geneticengine.random.sources import RandomSource
from geneticengine.representations.api import RepresentationWithCrossover, Representation
from geneticengine.evaluation import Evaluator
from typing import Iterator, Any, TypeVar

from sklearn.datasets import load_breast_cancer

import pandas as pd

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve, auc
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

import time

import seaborn as sns
import matplotlib.pyplot as plt

import os     

from functools import lru_cache

In [33]:
# model_used = DecisionTreeClassifier(random_state=42, max_depth=6,max_features='log2', min_samples_leaf=5, min_samples_split=10)
# model_used = LogisticRegression(random_state=42)
model_used = RandomForestClassifier(random_state=42, n_estimators=20, n_jobs=-1)
target_fpr_value = 0.05

### Functions and Data Preprocessing

In [34]:
df = pd.read_csv('base.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 32 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   fraud_bool                        1000000 non-null  int64  
 1   income                            1000000 non-null  float64
 2   name_email_similarity             1000000 non-null  float64
 3   prev_address_months_count         1000000 non-null  int64  
 4   current_address_months_count      1000000 non-null  int64  
 5   customer_age                      1000000 non-null  int64  
 6   days_since_request                1000000 non-null  float64
 7   intended_balcon_amount            1000000 non-null  float64
 8   payment_type                      1000000 non-null  object 
 9   zip_count_4w                      1000000 non-null  int64  
 10  velocity_6h                       1000000 non-null  float64
 11  velocity_24h                      1000

In [35]:
df.head(2)

,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0


In [36]:
df = df.sample(frac=0.2, random_state=42)
df.drop('month', axis=1, inplace=True)
print(len(df))

200000


In [37]:
categorical_features = [col for col in df.columns if df[col].dtype == 'object']

print(categorical_features)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

encoded_data = encoder.fit_transform(df[categorical_features])

encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_features))

df = df.drop(columns=categorical_features).reset_index(drop=True)
df = pd.concat([df, encoded_df], axis=1)

df.head(2)

['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,zip_count_4w,velocity_6h,...,housing_status_BE,housing_status_BF,housing_status_BG,source_INTERNET,source_TELEAPP,device_os_linux,device_os_macintosh,device_os_other,device_os_windows,device_os_x11
0,0,0.1,0.218119,110,7,20,0.015684,-1.013463,687,1442.159111,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0,0.1,0.373086,29,7,20,0.024277,19.342285,1230,6236.294791,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [38]:
# df['month'].value_counts().sort_index()

In [39]:
# X = df.drop(['fraud_bool'], axis=1)
# y = df['fraud_bool']

# X_train = X[X['month']<3]
# X_test = X[X['month']==3]
# y_train = y[X['month']<3]
# y_test = y[X['month']==3]

# X_train.drop('month', axis=1, inplace=True)
# X_test.drop('month', axis=1, inplace=True)

In [40]:
X = df.drop(['fraud_bool'], axis=1)
y = df['fraud_bool']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [41]:
print(y_train.value_counts(),y_test.value_counts())

fraud_bool
0    158313
1      1687
Name: count, dtype: int64 fraud_bool
0    39578
1      422
Name: count, dtype: int64


In [42]:
# features_to_scale = [col for col in X_train.select_dtypes(include=np.number).columns.tolist() if col not in encoded_df.columns]

# scaler = StandardScaler()
# X_train[features_to_scale] = scaler.fit_transform(X_train[features_to_scale])
# X_test[features_to_scale] = scaler.transform(X_test[features_to_scale])

### Baseline Model

In [43]:
feature_names = X_train.columns.tolist()
n_features = len(feature_names)

In [44]:
model = model_used

model.fit(X_train, y_train)

train_probs = model.predict_proba(X_train)[:,1]

fpr, tpr, thresholds = roc_curve(y_train, train_probs)

target_fpr = target_fpr_value

if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0] #where returns a tuple
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    train_tpr_at_fpr = tpr[best_index]

probs = model.predict_proba(X_test)[:,1]

fpr, tpr, thresholds = roc_curve(y_test, probs)


baseline_recall_at_fpr = 0.0

if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0] #where returns a tuple
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    baseline_tpr_at_fpr = tpr[best_index]

# plt.figure(figsize=(20,10))
# plot_tree(model, feature_names=feature_names, class_names=['No Fraud', 'Fraud'], filled=True)
# plt.savefig("tree_based.svg", format='svg')

print(f"Train TPR: {train_tpr_at_fpr}, Test TPR: {baseline_tpr_at_fpr}")

Train TPR: 1.0, Test TPR: 0.3341232227488152


### Feature Selection with and model performance

#### Feature Importance

In [45]:
# feature_importances = model.feature_importances_
# importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances}).sort_values(by='Importance', ascending=False)

# print(importance_df)

In [46]:
# importances = model.feature_importances_     
# labels = np.array(feature_names)             
                                             
# order = np.argsort(importances)[::-1]        
# sorted_importances = importances[order]      
# sorted_labels = labels[order]                

# plt.figure(figsize=(15, 6))                                
# plt.plot(range(1, len(sorted_importances) +  
# 1), sorted_importances, marker="o")          
# plt.xticks(range(1, len(sorted_labels) + 1), 
# sorted_labels, rotation=45, ha="right")      
# plt.ylabel("Importance")                     
# plt.xlabel("Ranked Features")                
# plt.tight_layout()                           
# plt.show()  

In [47]:
    # selected_features = importance_df.loc[importance_df['Importance'] >= 0.01, 'Feature']
    # X_train = X_train[selected_features]
    # X_test = X_test[selected_features]

    # feature_names = X_train.columns.tolist()
    # n_features = len(feature_names)

    # n_features

In [48]:
# corr_matrix = df.corr(method='pearson').abs()

# upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.9)]

# print(to_drop)


In [49]:
# importances = model.feature_importances_     
# labels = np.array(feature_names)             
                                             
# order = np.argsort(importances)[::-1]        
# sorted_importances = importances[order]      
# sorted_labels = labels[order]                

# plt.figure(figsize=(15, 6))                                
# plt.plot(range(1, len(sorted_importances) +  
# 1), sorted_importances, marker="o")          
# plt.xticks(range(1, len(sorted_labels) + 1), 
# sorted_labels, rotation=45, ha="right")      
# plt.ylabel("Importance")                     
# plt.xlabel("Ranked Features")                
# plt.tight_layout()                           
# plt.show()  

In [50]:
# selected_features = importance_df.loc[importance_df['Importance']>0, 'Feature']
# X_train = X_train[selected_features]
# X_test = X_test[selected_features]
# # X_train.shape[1]

# feature_names = X_train.columns.tolist()
# n_features = len(feature_names)

# n_features


#### RFE

In [51]:
# selector = RFE(model, n_features_to_select=0.8, step=1)
# selector.fit(X_train, y_train)

In [52]:
# ranking = selector.ranking_
# selected_features = [feature for feature, rank in zip(feature_names, ranking) if rank == 1]

# X_train = X_train[selected_features]
# X_test = X_test[selected_features]

# feature_names = X_train.columns.tolist()
# n_features = len(feature_names)

# n_features

#### Model Baseline

In [53]:
# model = model_used

# model.fit(X_train, y_train)

# probs = model.predict_proba(X_test)[:,1]

# fpr, tpr, thresholds = roc_curve(y_test, probs)

# target_fpr = target_fpr_value
# baseline_recall_at_fpr = 0.0

# if np.any(fpr <= target_fpr):
#     valid_indices = np.where(fpr <= target_fpr)[0] #where returns a tuple
#     best_index = valid_indices[np.argmax(tpr[valid_indices])]
#     baseline_tpr_at_fpr = tpr[best_index]

# print(f"TPR: {baseline_tpr_at_fpr}")

### Grammar

In [54]:
@dataclass
class Value(ABC):
    def evaluate(self):
        pass

class Scalar(ABC):
    pass

In [55]:
@weight(0.97)
@dataclass #Scalar Features (1)
class ScalarVar(Scalar): 
    index: Annotated[int, IntRange(0,n_features-1)]

    def evaluate(self, X_np):
        return X_np[:, self.index]
    
    def __str__(self):
        return feature_names[self.index]

In [56]:
#scalar -> scalar
@weight(0.01)
@dataclass 
class Add(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return self.left.evaluate(X_np) + self.right.evaluate(X_np)
    
    def __str__(self):
        return f"({self.left} + {self.right})"

@weight(0.01)
@dataclass
class Subtract(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return (self.left.evaluate(X_np)) - (self.right.evaluate(X_np))
    
    def __str__(self):
        return f"({self.left} - {self.right})"

@weight(0.01)
@dataclass
class Multiply(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return self.left.evaluate(X_np) * self.right.evaluate(X_np)
    
    def __str__(self):
        return f"({self.left} * {self.right})"

In [57]:
grammar = extract_grammar([Add, Subtract, Multiply,ScalarVar], Scalar)
print(f"Grammar: {repr(grammar)}")

Grammar: Grammar<Starting=Scalar,Productions={
Scalar -> Add(right: Scalar, left: Scalar)<0.01>|
	Subtract(right: Scalar, left: Scalar)<0.01>|
	Multiply(right: Scalar, left: Scalar)<0.01>|
	ScalarVar(index: Annotated[int])<0.97>
}


### Fitness and GP

In [58]:
ARCHIVE_TRAIN_DF = X_train.copy()
ARCHIVE_TEST_DF = X_test.copy()

ARCHIVE_TEMP : list[Individual] = []

In [59]:
X_train_np = ARCHIVE_TRAIN_DF.to_numpy()
X_test_np = ARCHIVE_TEST_DF.to_numpy()

def fitness_function(individual: Scalar): #individual -> expression
    start = time.perf_counter()
    train_feature = individual.evaluate(X_train_np)
    test_feature = individual.evaluate(X_test_np)
    if train_feature.ndim == 0:
        train_feature = np.full(X_train_np.shape[0], train_feature)
    if test_feature.ndim == 0:
        test_feature = np.full(X_test_np.shape[0], test_feature)

    X_train_augmented = np.c_[X_train_np, np.array(train_feature).reshape(-1,1)]
    X_test_augmented = np.c_[X_test_np, np.array(test_feature).reshape(-1,1)]

    model = model_used
    
    model.fit(X_train_augmented, y_train)
    
    probs = model.predict_proba(X_test_augmented)[:, 1]

    fpr, tpr, thresholds = roc_curve(y_test, probs)

    if np.any(fpr <= target_fpr):
        valid_indices = np.where(fpr<=target_fpr)[0]
        best_indice = valid_indices[np.argmax(tpr[valid_indices])]
        tpr_at_fpr = tpr[best_indice]

    tpr_diff = tpr_at_fpr-baseline_tpr_at_fpr
        
    features, num_operations = analyse_complexity(individual)

    end = time.perf_counter()
    elapsed = end - start
    return [tpr_at_fpr, tpr_diff, num_operations, elapsed]



In [60]:
def analyse_complexity(individual: Scalar):
    if isinstance(individual, ScalarVar):
        return {individual.index}, 0 #unique feature
    
    total_features = set()
    total_operations = 1
    if hasattr(individual, 'left') and hasattr(individual, 'right'):
        left_features, left_operations = analyse_complexity(individual.left)
        right_features, right_operations = analyse_complexity(individual.right)
        total_features.update(left_features)
        total_features.update(right_features)
        total_operations += left_operations + right_operations
    elif hasattr(individual, 'arr'):
        arr_features, arr_operations = analyse_complexity(individual.arr)
        total_features.update(arr_features)
        total_operations += arr_operations
    return total_features, total_operations


In [61]:
class ArchiveStep(GeneticStep):
    def iterate(
        self,
        problem: Problem,
        evaluator: Evaluator,
        representation: Representation,
        random: RandomSource,
        population: Iterator[PhenotypicIndividual],
        target_size: int,
        generation: int,
    ) -> Iterator[PhenotypicIndividual]:
        global ARCHIVE_TEMP, train_tpr_at_fpr, baseline_tpr_at_fpr, ARCHIVE_TRAIN_DF, ARCHIVE_TEST_DF, X_train_np, X_test_np
        best_fitness = 0
        for i, individual in enumerate(population):
            if individual.get_fitness(problem).fitness_components[0] > baseline_tpr_at_fpr and individual.get_fitness(problem).fitness_components[0] > best_fitness:
                best_fitness = individual.get_fitness(problem).fitness_components[0]
                print("New Individual:", str(individual.get_phenotype()), "Fitness:", individual.get_fitness(problem).fitness_components)
                if not ARCHIVE_TEMP:
                    ARCHIVE_TEMP.append(individual)
                else:
                    ARCHIVE_TEMP[0] = individual
            yield individual

        if ARCHIVE_TEMP:
            print(f"Archive Size: {len(ARCHIVE_TEMP)}")
            for ind in ARCHIVE_TEMP:
                train_feature_new = ind.get_phenotype().evaluate(X_train_np)
                test_feature_new = ind.get_phenotype().evaluate(X_test_np)

                ARCHIVE_TRAIN_DF[str(ind)] = train_feature_new
                ARCHIVE_TEST_DF[str(ind)] = test_feature_new

            ARCHIVE_TRAIN_DF = ARCHIVE_TRAIN_DF.loc[:, ~ARCHIVE_TRAIN_DF.columns.duplicated()]
            ARCHIVE_TEST_DF = ARCHIVE_TEST_DF.loc[:, ~ARCHIVE_TEST_DF.columns.duplicated()]

            ARCHIVE_TRAIN_DF.columns = [str(col) for col in ARCHIVE_TRAIN_DF.columns]
            ARCHIVE_TEST_DF.columns = [str(col) for col in ARCHIVE_TEST_DF.columns]

            model = model_used
            model.fit(ARCHIVE_TRAIN_DF, y_train)
            target_fpr = target_fpr_value
            probs = model.predict_proba(ARCHIVE_TEST_DF)[:,1]
            fpr, tpr, thresholds = roc_curve(y_test, probs)
            if np.any(fpr <= target_fpr):
                valid_indices = np.where(fpr <= target_fpr)[0]
                best_index = valid_indices[np.argmax(tpr[valid_indices])]
                baseline_tpr_at_fpr = tpr[best_index]
            print(f"Test TPR: {baseline_tpr_at_fpr}, Test shape: {ARCHIVE_TEST_DF.shape}")
            ARCHIVE_TEMP = []
            X_train_np = ARCHIVE_TRAIN_DF.to_numpy()
            X_test_np = ARCHIVE_TEST_DF.to_numpy()

In [62]:
def lexicase_step():
    return SequenceStep(
        ArchiveStep(),
        ParallelStep(
            [
                ElitismStep(),
                NoveltyStep(),
                SequenceStep(
                    LexicaseSelection(epsilon=True),
                    # TournamentSelection(tournament_size=3),
                    GenericCrossoverStep(0.9),
                    GenericMutationStep(0.1),
                )
            ],
            weights=[0.05, 0.05, 0.9]
        ),
    )

prob = MultiObjectiveProblem(
    fitness_function=fitness_function,
    minimize=[False, False, True, True],
)
r = NativeRandomSource(123)
alg = GeneticProgramming(
    problem=prob,
    budget=TimeBudget(1800),
    population_size=25,
    representation=TreeBasedRepresentation(grammar, MaxDepthDecider(r, grammar, 4)),
    random=r,
    step=lexicase_step(),
    tracker=ProgressTracker(
        prob,
        recorders=[CSVSearchRecorder(
            csv_path='output.csv', 
            problem=prob, 
            fields={
                    "Eval Time": lambda t,i,p: i.get_fitness(p).fitness_components[3],
                    "TPR Test": lambda t,i,p: i.get_fitness(p).fitness_components[0],
                    "TPR Test Diff": lambda t,i,p: i.get_fitness(p).fitness_components[1],
                    "Expression": lambda t, i, p: i.get_phenotype(),
                    "Num Operations": lambda t,i,p: i.get_fitness(p).fitness_components[2],
                    'Generation': lambda t,i,p: i.metadata["generation"]
                    },
            only_record_best_individuals=False)]
    )
    
)

solutions = alg.search()

New Individual: bank_months_count Fitness: [np.float64(0.3459715639810427), np.float64(0.0118483412322275), 0, 1.7311991999740712]
New Individual: session_length_in_minutes Fitness: [np.float64(0.35308056872037913), np.float64(0.018957345971563955), 0, 1.6810125999618322]
New Individual: source_INTERNET Fitness: [np.float64(0.36018957345971564), np.float64(0.026066350710900466), 0, 1.6171383999753743]
Archive Size: 1
Test TPR: 0.3341232227488152, Test shape: (40000, 51)


KeyboardInterrupt: 

In [ ]:
#create a copy of ARCHIVE_TRAIN_DF and ARCHIVE_TEST_DF to use later
ARCHIVE_TRAIN_DF_FINAL = ARCHIVE_TRAIN_DF.copy()
ARCHIVE_TEST_DF_FINAL = ARCHIVE_TEST_DF.copy()

In [ ]:
#train a model with the final augmented data
model = model_used
model.fit(ARCHIVE_TRAIN_DF_FINAL, y_train)
target_fpr = target_fpr_value
probs = model.predict_proba(ARCHIVE_TEST_DF_FINAL)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, probs)
if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    final_tpr_at_fpr = tpr[best_index]
print(f"Final Test TPR: {final_tpr_at_fpr}, Test shape: {ARCHIVE_TEST_DF_FINAL.shape}")

#remove the last column in both dataasets
ARCHIVE_TRAIN_DF_FINAL = ARCHIVE_TRAIN_DF_FINAL.iloc[:,:-1]
ARCHIVE_TEST_DF_FINAL = ARCHIVE_TEST_DF_FINAL.iloc[:,:-1]
model = model_used
model.fit(ARCHIVE_TRAIN_DF_FINAL, y_train)
target_fpr = target_fpr_value
probs = model.predict_proba(ARCHIVE_TEST_DF_FINAL)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, probs)
if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    final_tpr_at_fpr = tpr[best_index]
print(f"Final Test TPR: {final_tpr_at_fpr}, Test shape: {ARCHIVE_TEST_DF_FINAL.shape}")


Final Test TPR: 0.3222748815165877, Test shape: (40000, 56)
Final Test TPR: 0.36255924170616116, Test shape: (40000, 55)


In [ ]:
model = model_used
model.fit(ARCHIVE_TRAIN_DF, y_train)
train_probs = model.predict_proba(ARCHIVE_TRAIN_DF)[:,1]
fpr, tpr, thresholds = roc_curve(y_train, train_probs)
target_fpr = target_fpr_value
if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    train_tpr_at_fpr = tpr[best_index]
probs = model.predict_proba(ARCHIVE_TEST_DF)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, probs)
if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    baseline_tpr_at_fpr = tpr[best_index]
print(f"Test TPR: {baseline_tpr_at_fpr}, Test shape: {ARCHIVE_TEST_DF.shape}")
# print(f"Train TPR: {train_tpr_at_fpr}, Train shape: {ARCHIVE_TRAIN_DF.shape}")

Test TPR: 0.3125, Test shape: (150936, 51)


In [ ]:
# #make a copy of the dataframe ARCHIVE_TRAIN_DF and ARCHIVE_TEST_DF
# ARCHIVE_TRAIN_DF_final = ARCHIVE_TRAIN_DF.copy()
# ARCHIVE_TEST_DF_final = ARCHIVE_TEST_DF.copy()

In [ ]:


# def tpr_at_fpr(y_true, y_score, target_fpr=0.05):
#     fpr, tpr, thresholds = roc_curve(y_true, y_score)
#     if np.any(fpr <= target_fpr):
#         valid_indices = np.where(fpr <= target_fpr)[0]
#         best_index = valid_indices[np.argmax(tpr[valid_indices])]
#         return tpr[best_index]
#     else:
#         return 0.0

# def scorer(estimator, X, y):
#     probs = estimator.predict_proba(X)[:,1]
#     return tpr_at_fpr(y, probs, target_fpr=0.05)

# search = RandomizedSearchCV(
#     DecisionTreeClassifier(random_state=42),
#     param_distributions={
#         'max_depth': [4,6,8,10],
#         'min_samples_split': [2,5,10,20],
#         'min_samples_leaf': [1,2,4,8],
#         'max_features': ['sqrt', 'log2', None],
#         'criterion': ['gini', 'entropy']
#     },
#     n_iter=100,
#     scoring=scorer,
#     cv=2,
#     verbose=3,
# )

# search.fit(ARCHIVE_TRAIN_DF_final, y_train)

In [ ]:
# best_tree = search.best_estimator_
# best_score = search.best_score_
# print(best_tree, best_score)